## RAGAS Evaluation of the Chatbot component of FIONAA.


Create separate SGDs on for  company

calc the metrics


create a hybrid retriever with lexical search to compare


In [14]:

from operator import itemgetter

import nest_asyncio
from dotenv import load_dotenv
from langchain_classic.retrievers.ensemble import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_docling.loader import DoclingLoader

#Semantic search scoped to a single case
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.llms import LangchainLLMWrapper
from ragas.testset import TestsetGenerator

nest_asyncio.apply()  

load_dotenv()



True

In [77]:

loader = DoclingLoader(file_path="../data/synthesia/synthesia_companies_house_document.pdf")
raw_docs = loader.load()

2026-02-28 19:32:08,266 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2026-02-28 19:32:08,317 - INFO - Going to convert document batch...
2026-02-28 19:32:08,318 - INFO - Initializing pipeline for StandardPdfPipeline with options hash e15bc6f248154cc62f8db15ef18a8ab7
2026-02-28 19:32:08,323 - INFO - Auto OCR model selected ocrmac.
2026-02-28 19:32:08,327 - INFO - Accelerator device: 'mps'
2026-02-28 19:32:12,346 - INFO - Accelerator device: 'mps'
2026-02-28 19:32:13,038 - INFO - Processing document synthesia_companies_house_document.pdf
2026-02-28 19:32:54,159 - ERROR - Task exception was never retrieved
future: <Task finished name='Task-21854' coro=<as_completed.<locals>.sema_coro() done, defined at /Users/stevegoodman/dev/fionaa-be/.venv/lib/python3.12/site-packages/ragas/async_utils.py:75> exception=KeyboardInterrupt()>
Traceback (most recent call last):
  File "/Users/stevegoodman/dev/fionaa-be/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py", line 3701,

In [78]:
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

/var/folders/bz/qcwzc_9j6zgggtsc_fbjryy40000gp/T/ipykernel_97276/507413754.py:1: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
/var/folders/bz/qcwzc_9j6zgggtsc_fbjryy40000gp/T/ipykernel_97276/507413754.py:2: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())


In [ ]:

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
from ragas.testset.synthesizers import (
  SingleHopSpecificQuerySynthesizer,
)

query_distribution = [
    (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 1.0),
]
dataset = generator.generate_with_langchain_docs(raw_docs, testset_size=20, query_distribution=query_distribution)


In [23]:
dataset.to_pandas()

,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,What this 10933652 mean in the annual report a...,[ANNUAL REPORT AND FINANCIAL STATEMENTS FOR TH...,10933652 is the registered number shown in the...,Financial Analyst,POOR_GRAMMAR,LONG,single_hop_specific_query_synthesizer
1,When was Philip Karim Chopin appointed as a di...,[COMPANY INFORMATION\nDirectors\nSteffen Tjerr...,Philip Karim Chopin was appointed as a directo...,IFRS Compliance Accountant,PERFECT_GRAMMAR,MEDIUM,single_hop_specific_query_synthesizer
2,wher is Consolidated Statement of Changes in E...,"[CONTENTS\nGroup Strategic Report, Page = 1-2....",The Consolidated Statement of Changes in Equit...,Financial Analyst,MISSPELLED,SHORT,single_hop_specific_query_synthesizer
3,What is the significance of 31 December 2024 i...,[Introduction\nThe Directors present their str...,31 December 2024 marks the end of the financia...,IFRS Compliance Accountant,PERFECT_GRAMMAR,MEDIUM,single_hop_specific_query_synthesizer
4,"What were the Group's consolidated revenue, lo...",[Business review\nThe Group's principal activi...,"For the year ended 31 December 2023, the Group...",Financial Analyst,WEB_SEARCH_LIKE,MEDIUM,single_hop_specific_query_synthesizer
5,wat does the board do?,[Financial key performance indicators\nAnnual ...,The board agrees annual budgets in advance and...,Financial Compliance Officer,MISSPELLED,SHORT,single_hop_specific_query_synthesizer
6,"In the context of IFRS 15 compliance, how does...",[Principal risks and uncertainties\nThe key bu...,The term Group refers to the collective entity...,IFRS Compliance Accountant,PERFECT_GRAMMAR,LONG,single_hop_specific_query_synthesizer
7,How come the company gotta rely so much on thi...,[Hosting risk\nThe Group is primarily dependen...,The company is mainly dependent on third party...,Financial Analyst,POOR_GRAMMAR,LONG,single_hop_specific_query_synthesizer
8,How The Group gonna get hurt if they not get n...,[Customer relations and competition risk\nThe ...,The Group's future revenues and operating resu...,Financial Analyst,POOR_GRAMMAR,LONG,single_hop_specific_query_synthesizer
9,Did any single customer account for more than ...,[Liquidity risk\nThe Group currently is cash f...,No single customer accounted for more than 10%...,Financial Compliance Officer,PERFECT_GRAMMAR,MEDIUM,single_hop_specific_query_synthesizer


In [25]:
dataset.to_pandas().to_csv('synthesia_sdg.csv')

In [4]:
# Construct rag app
from vector_store import get_store


In [70]:

RAG_TEMPLATE = """\
You are a financial assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.
Be concise and specific.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

chat_model = ChatOpenAI(model="gpt-4.1-nano")
store = await get_store()
case_number="SteveGoodman"
retriever = store.as_retriever(
    search_kwargs={
        "filter": {"case_number": case_number},
        "k": 5
    }
)


In [63]:
naive_retrieval_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

result = await naive_retrieval_chain.ainvoke({"question" : "what are the key risks seen by this company?"})
result["response"].content

2026-02-28 18:05:50,307 - WARNING - Failed to send compressed multipart ingest: langsmith.utils.LangSmithRateLimitError: Rate limit exceeded for https://api.smith.langchain.com/runs/multipart. HTTPError('429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Too many requests: tenant exceeded usage limits: Monthly unique traces usage limit exceeded"}\n')trace=019ca56d-9975-7eb1-944a-e4d933cc5a35,id=019ca56d-9975-7eb1-944a-e4d933cc5a35; trace=019ca56d-9975-7eb1-944a-e4d933cc5a35,id=019ca56d-9981-7f13-a5f5-c4dbdaacf1d7; trace=019ca56d-9975-7eb1-944a-e4d933cc5a35,id=019ca56d-9982-7bf1-9667-a738617394a1; trace=019ca56d-9975-7eb1-944a-e4d933cc5a35,id=019ca56d-9983-7c60-aa41-84c515d20b74; trace=019ca56d-9975-7eb1-944a-e4d933cc5a35,id=019ca56d-9984-7de0-9fce-87ac143e42af; trace=019ca56d-9975-7eb1-944a-e4d933cc5a35,id=019ca56d-9983-7c60-aa41-84c515d20b74; trace=019ca56d-9975-7eb1-944a-e4d933cc5a35,id=019ca56d-9984-7de0-9fce-87ac143e42af; trace=

'Based on the provided documents, the key risks seen by the company are not explicitly detailed. However, some potential risks can be inferred:\n\n1. **Limited audit requirements** - The company has not been required to obtain an audit due to its small company status, which might pose risks related to the completeness and accuracy of financial reporting.\n\n2. **Control and Ownership Concentration** - The company is controlled entirely by the director who owns 100% of the share capital, which could pose risks related to governance, lack of checks and balances, and potential for mismanagement or conflicts of interest.\n\n3. **Financial Performance** - The company has shown a slight increase in turnover from £99,163 to £101,112, but also face relatively low profit margins with significant administrative and employment expenses, which might impact profitability and financial stability.\n\n4. **Dependence on a Single Control Person** - Since all control resides with one individual, there i

2026-02-28 18:05:55,307 - WARNING - Failed to send compressed multipart ingest: langsmith.utils.LangSmithRateLimitError: Rate limit exceeded for https://api.smith.langchain.com/runs/multipart. HTTPError('429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Too many requests: tenant exceeded usage limits: Monthly unique traces usage limit exceeded"}\n')trace=019ca56d-9975-7eb1-944a-e4d933cc5a35,id=019ca56d-a469-7d22-8bce-f60ee419de21; trace=019ca56d-9975-7eb1-944a-e4d933cc5a35,id=019ca56d-a45e-7563-8070-5a0b62e1c29b; trace=019ca56d-9975-7eb1-944a-e4d933cc5a35,id=019ca56d-a45e-7563-8070-59f96907eece; trace=019ca56d-9975-7eb1-944a-e4d933cc5a35,id=019ca56d-9975-7eb1-944a-e4d933cc5a35


In [87]:
case_number="Synthesia"


retriever = store.as_retriever(
    search_kwargs={
        "filter": {"case_number": case_number},
        "k": 5
    }
)

naive_retrieval_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

result = await naive_retrieval_chain.ainvoke({"question" : "what are the key risks seen by this company?"})
result["response"].content

2026-02-28 19:45:29,894 - WARNING - Failed to send compressed multipart ingest: langsmith.utils.LangSmithRateLimitError: Rate limit exceeded for https://api.smith.langchain.com/runs/multipart. HTTPError('429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Too many requests: tenant exceeded usage limits: Monthly unique traces usage limit exceeded"}\n')trace=019ca5c8-d799-7770-b1b9-2264e5a4fd41,id=019ca5c8-d799-7770-b1b9-2264e5a4fd41; trace=019ca5c8-d799-7770-b1b9-2264e5a4fd41,id=019ca5c8-d7a7-7a22-ba87-82f008ca1413; trace=019ca5c8-d799-7770-b1b9-2264e5a4fd41,id=019ca5c8-d7a8-7c60-8b84-9f25c2376500; trace=019ca5c8-d799-7770-b1b9-2264e5a4fd41,id=019ca5c8-d7a9-76c2-a7ed-666ac486a0b2; trace=019ca5c8-d799-7770-b1b9-2264e5a4fd41,id=019ca5c8-d7a9-76c2-a7ed-6679c5b954a6; trace=019ca5c8-d799-7770-b1b9-2264e5a4fd41,id=019ca5c8-d7a9-76c2-a7ed-666ac486a0b2; trace=019ca5c8-d799-7770-b1b9-2264e5a4fd41,id=019ca5c8-d7a9-76c2-a7ed-6679c5b954a6; trace=

'The key risks include customer relation and competition risks, financial risks such as credit and liquidity risks, and the reliance on external capital due to current cash flow negativity.'

2026-02-28 19:45:34,780 - WARNING - Failed to send compressed multipart ingest: langsmith.utils.LangSmithRateLimitError: Rate limit exceeded for https://api.smith.langchain.com/runs/multipart. HTTPError('429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Too many requests: tenant exceeded usage limits: Monthly unique traces usage limit exceeded"}\n')trace=019ca5c8-d799-7770-b1b9-2264e5a4fd41,id=019ca5c8-e59e-7bf3-ab37-0158d103c28c; trace=019ca5c8-d799-7770-b1b9-2264e5a4fd41,id=019ca5c8-e595-7c20-a8ac-c4a93dd5742a; trace=019ca5c8-d799-7770-b1b9-2264e5a4fd41,id=019ca5c8-e595-7c20-a8ac-c497d78bbb3e; trace=019ca5c8-d799-7770-b1b9-2264e5a4fd41,id=019ca5c8-d799-7770-b1b9-2264e5a4fd41


In [33]:
from ragas import EvaluationDataset, RunConfig, evaluate
from ragas.metrics import (
  ContextEntityRecall,
  ContextPrecision,
  FactualCorrectness,
  Faithfulness,
  LLMContextRecall,
)

custom_run_config = RunConfig(timeout=360)
evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))


async def aevaluate_retreiver(candidate_retreiver, retriever_name):
  for test_row in dataset:
    response = await candidate_retreiver.ainvoke({"question" : test_row.eval_sample.user_input})
    test_row.eval_sample.response = response["response"].content
    test_row.eval_sample.retrieved_contexts = [context.page_content for context in response["context"]]

  evaluation_dataset = EvaluationDataset.from_pandas(dataset.to_pandas())

  result = evaluate(
      dataset=evaluation_dataset,
      metrics=[ContextPrecision(), LLMContextRecall(), Faithfulness(), FactualCorrectness(),  ContextEntityRecall()],
      llm=evaluator_llm,
      run_config=custom_run_config
  )
  return result 

baseline_synthesia_result = await aevaluate_retreiver(naive_retrieval_chain, 'naive_retriever')

baseline_synthesia_result

/var/folders/bz/qcwzc_9j6zgggtsc_fbjryy40000gp/T/ipykernel_97276/1263718308.py:1: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextPrecision
  from ragas.metrics import ContextPrecision, LLMContextRecall, Faithfulness, FactualCorrectness, ContextEntityRecall
/var/folders/bz/qcwzc_9j6zgggtsc_fbjryy40000gp/T/ipykernel_97276/1263718308.py:1: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import ContextPrecision, LLMContextRecall, Faithfulness, FactualCorrectness, ContextEntityRecall
/var/folders/bz/qcwzc_9j6zgggtsc_fbjryy40000gp/T/ipykernel_97276/1263718308.py:1: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is depre

Evaluating:   0%|          | 0/100 [00:00<?, ?it/s]

2026-02-28 15:42:11,487 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-28 15:42:11,488 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-28 15:42:11,489 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-28 15:42:13,068 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-28 15:42:13,069 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-28 15:42:13,070 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-28 15:42:13,071 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-28 15:42:13,072 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-28 15:42:13,073 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "

{'context_precision': 0.7167, 'context_recall': 0.8750, 'faithfulness': 0.7461, 'factual_correctness(mode=f1)': 0.5055, 'context_entity_recall': 0.5713}

In [89]:
# Hybrid retriever: BM25 (sparse) + pgvector dense search combined via EnsembleRetriever
# BM25Retriever operates on an in-memory document list, so we first fetch all chunks
# for the case from pgvector, then hand them to BM25.


case_number = "Synthesia"

# --- 1. Fetch all documents for this case to seed BM25 ---
# We use a broad similarity search with a high k; for production you'd page through
# all rows, but for evaluation this is sufficient.
all_docs = await store.asimilarity_search(
    "company overview",
    k=200,
    filter={"case_number": case_number},
)
print(f"Fetched {len(all_docs)} docs for BM25 corpus")

# --- 2. Sparse retriever (BM25) ---
bm25_retriever = BM25Retriever.from_documents(all_docs, k=5)

# --- 3. Dense retriever (pgvector cosine similarity) ---
dense_retriever = store.as_retriever(
    search_kwargs={"filter": {"case_number": case_number}, "k": 5}
)

# --- 4. Ensemble: equal weights, deduplication via Reciprocal Rank Fusion ---
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, dense_retriever],
    weights=[0.5, 0.5],
)

# Smoke test
test_docs = await hybrid_retriever.ainvoke("what are the key risks seen by this company?")
print(f"Hybrid retriever returned {len(test_docs)} docs")
for d in test_docs[:2]:
    print(f"  [{d.metadata.get('chunk_type', '?')} p{d.metadata.get('page_num', '?')}] {d.page_content[:120]}...")

2026-02-28 19:51:53,132 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


Fetched 200 docs for BM25 corpus


2026-02-28 19:51:53,952 - WARNING - Failed to send compressed multipart ingest: langsmith.utils.LangSmithRateLimitError: Rate limit exceeded for https://api.smith.langchain.com/runs/multipart. HTTPError('429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Too many requests: tenant exceeded usage limits: Monthly unique traces usage limit exceeded"}\n')trace=019ca5ce-b493-7dd1-b14b-f5456e1fd3ac,id=019ca5ce-b493-7dd1-b14b-f5456e1fd3ac; trace=019ca5ce-b493-7dd1-b14b-f5456e1fd3ac,id=019ca5ce-b495-7530-af4a-8a8f1524c9bf; trace=019ca5ce-b493-7dd1-b14b-f5456e1fd3ac,id=019ca5ce-b496-77e2-bd1e-b5940f3d104d; trace=019ca5ce-b493-7dd1-b14b-f5456e1fd3ac,id=019ca5ce-b495-7530-af4a-8a8f1524c9bf
2026-02-28 19:51:55,643 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


Hybrid retriever returned 9 docs
  [chunkText p3] <a id='ad1fddc4-f074-40cc-8e5b-f1cb14cd1011'></a>

**Principal risks and uncertainties**
The key business risks that cou...
  [chunkText p9] <a id='832ddbc3-b718-46d0-88af-d2b338435ffc'></a>

Our responsibilities and the responsibilities of the directors with r...


2026-02-28 19:51:56,360 - WARNING - Failed to send compressed multipart ingest: langsmith.utils.LangSmithRateLimitError: Rate limit exceeded for https://api.smith.langchain.com/runs/multipart. HTTPError('429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Too many requests: tenant exceeded usage limits: Monthly unique traces usage limit exceeded"}\n')trace=019ca5ce-b493-7dd1-b14b-f5456e1fd3ac,id=019ca5ce-b496-77e2-bd1e-b5940f3d104d; trace=019ca5ce-b493-7dd1-b14b-f5456e1fd3ac,id=019ca5ce-b493-7dd1-b14b-f5456e1fd3ac


In [90]:
# Wrap hybrid_retriever in the same RAG chain shape used by aevaluate_retreiver
hybrid_retrieval_chain = (
    {"context": itemgetter("question") | hybrid_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

hybrid_synthesia_result = await aevaluate_retreiver(hybrid_retrieval_chain, "hybrid_retriever")
hybrid_synthesia_result

2026-02-28 19:52:09,077 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-02-28 19:52:09,924 - WARNING - Failed to send compressed multipart ingest: langsmith.utils.LangSmithRateLimitError: Rate limit exceeded for https://api.smith.langchain.com/runs/multipart. HTTPError('429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Too many requests: tenant exceeded usage limits: Monthly unique traces usage limit exceeded"}\n')trace=019ca5ce-f17c-7d62-bb88-74d6eab3d86c,id=019ca5ce-f17c-7d62-bb88-74d6eab3d86c; trace=019ca5ce-f17c-7d62-bb88-74d6eab3d86c,id=019ca5ce-f17e-7493-b880-b838dab02edf; trace=019ca5ce-f17c-7d62-bb88-74d6eab3d86c,id=019ca5ce-f17f-7f73-9747-7424a3916836; trace=019ca5ce-f17c-7d62-bb88-74d6eab3d86c,id=019ca5ce-f180-7d73-98d0-363186027aec; trace=019ca5ce-f17c-7d62-bb88-74d6eab3d86c,id=019ca5ce-f181-74f2-ad10-da1a6b35319c; trace=019ca5ce-f17c-7d62-bb88-74d6eab3d86c,id=019ca5ce-f181-74f2-ad

Evaluating:   0%|          | 0/100 [00:00<?, ?it/s]

2026-02-28 19:52:35,406 - WARNING - Failed to send compressed multipart ingest: langsmith.utils.LangSmithRateLimitError: Rate limit exceeded for https://api.smith.langchain.com/runs/multipart. HTTPError('429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Too many requests: tenant exceeded usage limits: Monthly unique traces usage limit exceeded"}\n')trace=019ca5cf-5146-78f2-ac0f-cb2fc77ebd1b,id=019ca5cf-51ff-7c42-abbe-ebce899fc6e7; trace=019ca5cf-5146-78f2-ac0f-cb2fc77ebd1b,id=019ca5cf-51fd-7dd1-8b82-aa8f7bf93f04; trace=019ca5cf-5146-78f2-ac0f-cb2fc77ebd1b,id=019ca5cf-51fd-7dd1-8b82-aa7717c76a20; trace=019ca5cf-5146-78f2-ac0f-cb2fc77ebd1b,id=019ca5cf-5146-78f2-ac0f-cb2fc77ebd1b; trace=019ca5cf-5684-70f2-ad18-9a0782dbc12f,id=019ca5cf-5684-70f2-ad18-9a0782dbc12f; trace=019ca5cf-5684-70f2-ad18-9a0782dbc12f,id=019ca5cf-5688-7c90-a94b-a7fe04a5fa9a; trace=019ca5cf-5684-70f2-ad18-9a0782dbc12f,id=019ca5cf-5688-7c90-a94b-a806bc8da344; trace=

{'context_precision': 0.4853, 'context_recall': 0.9500, 'faithfulness': 0.7922, 'factual_correctness(mode=f1)': 0.6755, 'context_entity_recall': 0.5595}

2026-02-28 19:55:42,677 - WARNING - Failed to send compressed multipart ingest: langsmith.utils.LangSmithRateLimitError: Rate limit exceeded for https://api.smith.langchain.com/runs/multipart. HTTPError('429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Too many requests: tenant exceeded usage limits: Monthly unique traces usage limit exceeded"}\n')trace=019ca5cf-5684-70f2-ad18-9a0782dbc12f,id=019ca5d2-254c-7181-8241-b6c976ba35ae; trace=019ca5cf-5684-70f2-ad18-9a0782dbc12f,id=019ca5d2-254a-71c3-aaef-c96acce25236; trace=019ca5cf-5684-70f2-ad18-9a0782dbc12f,id=019ca5d1-88ca-7bc3-b975-c9adbe8bb4c9; trace=019ca5cf-5684-70f2-ad18-9a0782dbc12f,id=019ca5d2-2745-7fa1-b5e8-91b0bb78c379; trace=019ca5cf-5684-70f2-ad18-9a0782dbc12f,id=019ca5d2-2744-70c1-bd35-32a439af49f5; trace=019ca5cf-5684-70f2-ad18-9a0782dbc12f,id=019ca5d1-67e9-7202-a28d-9d5d991701e1; trace=019ca5cf-5684-70f2-ad18-9a0782dbc12f,id=019ca5d2-293e-7321-99a2-5c74e1c71ef2; trace=

In [ ]:
{'context_precision': 0.7167, 'context_recall': 0.8750, 'faithfulness': 0.7461, 'factual_correctness(mode=f1)': 0.5055, 'context_entity_recall': 0.5713}


In [95]:
from pandas import DataFrame

results_df = DataFrame([eval(baseline_synthesia_result.__repr__()), 

eval(hybrid_synthesia_result.__repr__()) ], 
index=['naive_retriever', 
'sparse_and_dense_retriever'])

results_df 

,context_precision,context_recall,faithfulness,factual_correctness(mode=f1),context_entity_recall
naive_retriever,0.7167,0.875,0.7461,0.5055,0.5713
sparse_and_dense_retriever,0.4853,0.950,0.7922,0.6755,0.5595


In [ ]:
#test this - this is our actual application
async def search_documents(query: str, case_number) -> str:
    """Search the parsed document chunks for this case using semantic similarity.

    Use this tool to find relevant passages from the documents uploaded for the
    current case (bank statements, annual accounts, etc.).

    Args:
        query: Natural language question or search phrase.
    """
    store = await get_store()
    docs = await store.asimilarity_search(
        query,
        k=5,
        filter={"case_number": case_number},
    )

    if not docs:
        return f"No relevant document chunks found for case '{case_number}'."

    parts = []
    for i, doc in enumerate(docs, 1):
        meta = doc.metadata
        header = (
            f"[Chunk {i} | type={meta.get('chunk_type', '?')} "
            f"| page={meta.get('page_num', '?')}]"
        )
        parts.append(f"{header}\n{doc.page_content}")

    return "\n\n---\n\n".join(parts)

<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>context_precision</th>
      <th>context_recall</th>
      <th>faithfulness</th>
      <th>factual_correctness(mode=f1)</th>
      <th>context_entity_recall</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>naive_retriever</th>
      <td>0.7167</td>
      <td>0.875</td>
      <td>0.7461</td>
      <td>0.5055</td>
      <td>0.5713</td>
    </tr>
    <tr>
      <th>sparse_and_dense_retriever</th>
      <td>0.4853</td>
      <td>0.950</td>
      <td>0.7922</td>
      <td>0.6755</td>
      <td>0.5595</td>
    </tr>
  </tbody>
</table>
</div>

'|                            |   context_precision |   context_recall |   faithfulness |   factual_correctness(mode=f1) |   context_entity_recall |\n|:---------------------------|--------------------:|-----------------:|---------------:|-------------------------------:|------------------------:|\n| naive_retriever            |              0.7167 |            0.875 |         0.7461 |                         0.5055 |                  0.5713 |\n| sparse_and_dense_retriever |              0.4853 |            0.95  |         0.7922 |                         0.6755 |                  0.5595 |'

Musings on possible improvements

Hybrid search for questions such as dates or assets?